# Vorkonditioniertes Gradientenverfahren

Dieses Notebook enthält die Erklärungen zu Algorithmus 4.11 (Armijo-Schrittweitensuche) und Algorithmus 4.12 (exakte Schrittweite für quadratische Funktionen). Im Browser-Beispiel kannst du zwischen der quadratischen Funktion, Himmelblau und Rosenbrock wählen.

Für die Python-Module in `src` installiere bei Bedarf die Pakete aus `requirements-lab.txt`. Die Browser-Ansicht braucht außerdem eine Internetverbindung, weil Plotly von einem CDN geladen wird.

## Grundidee: Vorkonditionierung als Geometriewahl

Wir minimieren eine differenzierbare Funktion $f:\mathbb{R}^n\to\mathbb{R}$. Am Punkt $x_k$ ist $g_k=\nabla f(x_k)$ die Richtung des stärksten lokalen Anstiegs in der euklidischen Geometrie. Eine symmetrisch positiv definite Matrix $M$ definiert eine andere Geometrie. Die zugehörige Abstiegsrichtung ist

$$
d_k=-M^{-1}g_k,\qquad g_k^T d_k=-g_k^T M^{-1}g_k<0\quad (g_k\ne 0).
$$

Intuitiv skaliert $M^{-1}$ die Komponenten des Gradienten: In Richtungen mit stärkerer Krümmung wird der Schritt anders gewichtet, sodass lang gestreckte Höhenlinien ausgeglichener werden können. Positive Definitheit ist wesentlich: Sie garantiert, dass $d_k$ tatsächlich eine Abstiegsrichtung ist. In der Implementierung wird kein Matrixinverses gebildet; `numpy.linalg.solve(M, g)` berechnet stabil die Lösung des linearen Gleichungssystems $Mz=g$.

### Algorithmus 4.11 — Armijo-Schrittweitensuche

1. Wähle $x_0$, ein SPD-Vorkonditionierer $M$ und Parameter $0<\sigma,\beta<1$.
2. Berechne $g_k=\nabla f(x_k)$ und löse $Mz_k=g_k$; setze $d_k=-z_k$.
3. Beginne mit einer positiven Probesprungweite $t$. Solange
   $$f(x_k+t d_k)>f(x_k)+\sigma t\,g_k^T d_k,$$
   verkleinere sie auf $\beta t$.
4. Übernimm den ersten Schritt, der die Armijo-Bedingung erfüllt: $x_{k+1}=x_k+t d_k$.
5. Wiederhole, bis der Gradient klein genug ist, die Änderung klein genug ist oder ein Iterationslimit greift.

Die rechte Seite der Armijo-Ungleichung ist die Tangente an $t\mapsto f(x_k+t d_k)$ bei $t=0$, mit einem Anteil $\sigma$ des erwarteten linearen Abstiegs. Die Bedingung verhindert, dass ein zu großer Schritt den tatsächlichen Funktionswert zu wenig senkt. Das geometrische Halbieren mit $\beta$ ist ein einfaches Zurückgehen, bis die lokale lineare Vorhersage hinreichend gut ist.

### Algorithmus 4.12 — quadratische Zielfunktion mit exakter Schrittweite (4.15)

Für

$$
f(x)=\tfrac12 x^TQx+c^Tx+\gamma,\qquad Q=Q^T\succ0,
$$
ist $\nabla f(x)=Qx+c$. Auf einer Suchgeraden $x_k+t d_k$ ist $\varphi_k(t)=f(x_k+t d_k)$ eine eindimensionale quadratische Funktion. Ihre Ableitungen lauten

$$
\varphi_k'(0)=g_k^T d_k<0,\qquad \varphi_k''(t)=d_k^TQd_k>0.
$$

Das eindimensionale Minimum liegt daher bei

$$
t_k=-\frac{\varphi_k'(0)}{\varphi_k''(0)}
    =-\frac{g_k^Td_k}{d_k^TQd_k}>0.
$$

Die Implementierung berechnet dieselbe Weite aus $\varphi(0)$, $\varphi'(0)$ und $\varphi(1)$: Für $\varphi(t)=\varphi(0)+\varphi'(0)t+a t^2$ ist $a=\varphi(1)-\varphi(0)-\varphi'(0)$ und damit $t=-\varphi'(0)/(2a)$. Anders als bei Armijo wird nicht ausprobiert und zurückgerudert; bei einer quadratischen Funktion ist das Minimum der Schnittkurve direkt bestimmbar.

Bei quadratischen Problemen beeinflusst auch die verallgemeinerte Konditionszahl $\kappa$ des Paares $(Q,M)$ die Konvergenz. Ein gut gewähltes $M$ kann die Geometrie ausgleichen. Die bekannte Schranke mit Faktor $((\kappa-1)/(\kappa+1))^2$ bezieht sich auf den Fehler der Funktionswerte bei exakter Schrittweite; sie ist keine garantierte Rate für Armijo.

## Interaktives Beispiel im Browser

Die Seite zeigt sechs getrennte Ansichten: Höhenlinien und Iterationswege, die Zielfläche, den Verlauf der Variablen, die Armijo-Schrittweitensuche, die verallgemeinerten Eigenwerte und eine Ergebnistabelle. Spektren und Konditionszahlen sind nur für die quadratische Funktion definiert.

### Zielfunktion und Bedienung

- Über **Zielfunktion** kannst du zwischen der quadratischen Funktion, Himmelblau und Rosenbrock aus `src/optimization_examples.py` wählen. Die Browser-Seite verwendet JavaScript-Nachbildungen dieser Python-Funktionen; sie importiert die Python-Datei nicht.
- Bei der quadratischen Funktion legen `H11`, `H12`, `H22`, `b1`, `b2` und `c` die Funktion fest. `H12` wird für beide Nebendiagonaleinträge verwendet und `H` muss positiv definit sein.
- Für Rosenbrock lassen sich die positiven Parameter `a` und `b` ändern. Himmelblau hat keine zusätzlichen Funktionsparameter.
- `M11`, `M12` und `M22` legen das Innenprodukt $M=\begin{pmatrix}M_{11}&M_{12}\\M_{12}&M_{22}\end{pmatrix}$ fest. `M12` wird gespiegelt; $M$ muss positiv definit sein.
- Beim Wechsel der Funktion wird ein dazu passender Startpunkt gesetzt. `x01` und `x02` kannst du danach anpassen. Plotbereich, Armijo-Parameter und Iterationsgrenzen bleiben einstellbar.
- Mit **Neu berechnen** werden die Eingaben übernommen. Über Metrikauswahl, Schieberegler und Schritttasten lässt sich der Verlauf untersuchen. In den Diagrammen kann gezoomt werden; die 3D-Fläche lässt sich drehen.

Bei Himmelblau und Rosenbrock wird der beste Endpunkt der drei Metrik-Läufe markiert. Da beide Funktionen mehrere lokale Minima haben können, ist das nicht notwendigerweise das globale Minimum. Die dritte Metrik ist dort eine positive Diagonalmatrix aus den Beträgen der Hessendiagonale am Startpunkt.

### Wie die Python-Dateien dazu passen

Die Erklärungen oben beziehen sich auf die Python-Implementierungen in `src/gradient_methods.py`: Dort gibt es sowohl die Armijo-Suche aus Algorithmus 4.11 als auch die exakte Schrittweite für quadratische Schnittfunktionen aus Algorithmus 4.12. `src/optimization_examples.py` enthält die Zielfunktionen und Ableitungen. `src/optimization_lab.py` ist eine separate Matplotlib-Version des quadratischen Vergleichs.

Die eingebettete Browser-Seite führt diese Python-Dateien nicht aus. Sie bildet die Funktionen und das Armijo-Verfahren eigenständig mit JavaScript nach; Änderungen an den Python-Funktionen müssen daher bei Bedarf auch in `Interactive_Optimization_Lab.html` nachvollzogen werden.

Führe die Codezelle aus, um die aktuelle HTML-Seite hier im Notebook zu öffnen. Falls die Notebook-Ansicht den eingebetteten Inhalt blockiert, kannst du über den Link unter der Ansicht die HTML-Datei direkt öffnen.

In [ ]:
from html import escape
from pathlib import Path
from IPython.display import HTML, display

html_file = next(
    (folder / "Interactive_Optimization_Lab.html"
     for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "Interactive_Optimization_Lab.html").is_file()),
    None,
)
if html_file is None:
    raise FileNotFoundError(
        "Interactive_Optimization_Lab.html wurde vom aktuellen Ordner aus nicht gefunden."
    )

html_document = html_file.read_text(encoding="utf-8")
iframe = (
    '<iframe title="Interaktives Gradientenverfahren" '
    'style="width:100%;height:1400px;border:1px solid #344357;border-radius:8px" '
    f'srcdoc="{escape(html_document, quote=True)}"></iframe>'
    f'<p><a href="{html_file.resolve().as_uri()}" target="_blank" '
    'rel="noopener">HTML-Datei in einem eigenen Browserfenster öffnen</a></p>'
)
display(HTML(iframe))